# ArgLLM baseline (DF-QuAD) — Gemma 4 31B on COM2SENSE (500 claims, earlier sample from the training split)

This is the **earlier adaptation** of the ArgLLM prompts described in Section 5.1 of the paper: the model writes its own facts, and the argument prompt asks for one argument on each side instead of allowing the model to decline. Strengths are computed with the authors' DF-QuAD implementation (Uncertainpy). Results: 0.5 Base 70.6, Est. Base 78.6.

Run on Kaggle in batches of 100, with intermediate results after each batch; the API key is read from Kaggle secrets (`OPENROUTER_API_KEY`).

In [5]:
!pip install -q openai datasets scikit-learn
import numpy as np, pandas as pd, pickle
from pathlib import Path
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from datasets import load_dataset

WORK_DIR = Path("/kaggle/working/gemma_com2sense"); WORK_DIR.mkdir(parents=True, exist_ok=True)
client = OpenAI(api_key=UserSecretsClient().get_secret("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")
MODEL_NAME = "google/gemma-4-31b-it:free"

ds = load_dataset("tasksource/com2sense", split="train")
df = ds.to_pandas()
df["gold"] = df["label"].apply(lambda x: 1 if str(x).strip().lower()=="true" else 0)
df["claim"] = df["sent"].astype(str)
true_ex = df[df["gold"]==1].sample(250, random_state=42)
false_ex = df[df["gold"]==0].sample(250, random_state=42)
baf_df = pd.concat([true_ex, false_ex]).sample(frac=1, random_state=42).reset_index(drop=True)
baf_df["example_index"] = range(len(baf_df))

def generate_facts(claim):
    prompt = f"""You are building atomic factual building blocks.
For the claim below, produce 3-5 SHORT, SEPARATE factual statements. Each must be:
- A single atomic fact about ONE concept in the claim.
- General commonsense knowledge, stated on its own.
- NOT a comparison, ranking, or conclusion. NOT about whether the claim is true or false.
Claim: "{claim}"
Return 3-5 atomic facts, one per line. Do not conclude."""
    import time as _t
    for _ in range(5):
        try:
            resp=client.chat.completions.create(model=MODEL_NAME,
                messages=[{"role":"system","content":"You output atomic facts. You never conclude."},
                          {"role":"user","content":prompt}], temperature=0.3, max_tokens=400)
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e): _t.sleep(12); continue
            return ""
    return ""
print("Setup done. First claim:", baf_df.iloc[0]["claim"][:50])

README.md:   0%|          | 0.00/745 [00:00<?, ?B/s]

train.json: 0.00B [00:00, ?B/s]

dev.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1608 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/782 [00:00<?, ? examples/s]

Setup done. First claim: Tim was happy that November was coming up since he


In [6]:
import os, sys, glob, re, time
if not glob.glob("/kaggle/working/**/Uncertainpy/src", recursive=True):
    os.system("cd /kaggle/working && git clone -q https://github.com/CLArg-group/argumentative-llms.git")
UNC = glob.glob("/kaggle/working/**/Uncertainpy/src", recursive=True)[0]
sys.path.insert(0, UNC)
from uncertainpy.gradual import BAG, Argument
from uncertainpy.gradual.algorithms.Acyclic import computeStrengthValues
from uncertainpy.gradual.semantics.modular import ProductAggregation, LinearInfluence
agg_f = ProductAggregation(); inf_f = LinearInfluence(conservativeness=1)

def llm_raw(prompt, mt=200):
    for _ in range(5):
        try:
            r=client.chat.completions.create(model=MODEL_NAME,messages=[{"role":"user","content":prompt}],temperature=0.7,max_tokens=mt,top_p=0.95)
            return r.choices[0].message.content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e): time.sleep(12); continue
            time.sleep(3)
    return ""
def am_prompt(claim, facts, support):
    verb="supporting" if support else "attacking"; rel="support" if support else "attack"
    return (f'Please provide a single short argument {verb} the following claim, using the facts as context. '
            f'Construct the argument so it refers to the truthfulness of the claim. '
            f'You must always produce a {rel} argument, even if weak; do not return N/A.\n'
            f'Facts:\n{facts}\n\nClaim: {claim}\nArgument:')
def ue_arg_prompt(argument, claim, support):
    favour="in favour of" if support else "against"; rel="supports" if support else "refutes"
    return (f'You are an analyst evaluating arguments. For the argument:\n\n"{argument}"\n\n'
            f'give your confidence it makes a compelling case {favour}:\n\n"{claim}"\n\n'
            f'Base it on how well the argument {rel} the statement and its accuracy. Respond 0-100%. Format:\nLikelihood:')
def ue_topic_prompt(claim, facts):
    return (f'You are an analyst. Using these facts as context:\n{facts}\n\nFor the statement:\n\n"{claim}"\n\n'
            f'give your confidence it is correct and truthful, 0-100%. Format:\nLikelihood:')
def gen_argument(claim, facts, support):
    for _ in range(2):
        txt=llm_raw(am_prompt(claim,facts,support),150); txt=re.sub(r'\*+','',txt); txt=re.sub(r'^\s*Argument\s*:\s*','',txt,flags=re.I).strip()
        lines=[l.strip() for l in txt.split("\n") if l.strip()]; txt=lines[0] if lines else txt
        if txt and "N/A" not in txt: return txt.strip('"').strip()
    return "N/A"
def get_lik(prompt):
    txt=llm_raw(prompt,150); m=re.search(r'Likelihood:\s*([0-9]{1,3})\s*%',txt) or re.search(r'([0-9]{1,3})\s*%',txt)
    return max(0,min(1,int(m.group(1))/100)) if m else 0.5
def arg_base(a,c,s): return None if a=="N/A" else get_lik(ue_arg_prompt(a,c,s))
def topic_base(c,f): return get_lik(ue_topic_prompt(c,f))
print("✅ Gemma DF-QuAD ready")

✅ Gemma DF-QuAD ready


In [7]:
DFQ_CKPT = WORK_DIR / "gemma_dfquad_ckpt.pkl"
def load_dfq(): return pickle.load(open(DFQ_CKPT,"rb")) if DFQ_CKPT.exists() else {}
def save_dfq(d): pickle.dump(d, open(DFQ_CKPT,"wb"))

def run_dfquad_batch(n=100):
    done = load_dfq()
    todo = [i for i in range(len(baf_df)) if i not in done][:n]
    print(f"Have {len(done)}/{len(baf_df)}. Running {len(todo)} (Gemma DF-QuAD, rate-limited)...")
    for k, idx in enumerate(todo, 1):
        row = baf_df.iloc[idx]
        claim = row["claim"]
        try:
            facts = generate_facts(claim)
            sup = gen_argument(claim, facts, True)
            att = gen_argument(claim, facts, False)
            res = {"gold": int(row["gold"]), "claim": claim, "facts": facts,
                   "sup": sup, "att": att,
                   "sup_base": arg_base(sup, claim, True),
                   "att_base": arg_base(att, claim, False),
                   "topic_base": topic_base(claim, facts)}
        except Exception as e:
            print(f"error at {idx}: {str(e)[:50]}"); continue
        d = load_dfq(); d[idx] = res; save_dfq(d)
        if k % 10 == 0 or k == len(todo):
            print(f"  {k}/{len(todo)} → {len(load_dfq())} total")
        time.sleep(0.3)
    print(f"✅ {len(load_dfq())}/{len(baf_df)} done")

run_dfquad_batch(100)

Have 0/500. Running 100 (Gemma DF-QuAD, rate-limited)...
  10/100 → 10 total
  20/100 → 20 total
  30/100 → 30 total
  40/100 → 40 total
  50/100 → 50 total
  60/100 → 60 total
  70/100 → 70 total
  80/100 → 80 total
  90/100 → 90 total
  100/100 → 100 total
✅ 100/500 done


In [11]:
run_dfquad_batch(100)

Have 100/500. Running 100 (Gemma DF-QuAD, rate-limited)...
  10/100 → 110 total
  20/100 → 120 total
  30/100 → 130 total
  40/100 → 140 total
  50/100 → 150 total
  60/100 → 160 total
  70/100 → 170 total
  80/100 → 180 total
  90/100 → 190 total
  100/100 → 200 total
✅ 200/500 done


In [14]:
def dfquad_pred(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

dfq = load_dfq(); idxs = sorted(dfq); gold = [dfq[i]["gold"] for i in idxs]; n = len(idxs)
print(f"=== GEMMA DF-QuAD on COM2SENSE ({n} examples) ===\n")
for est, lab in [(False,"0.5 Base (fair)"), (True,"Est. Base")]:
    pred = [int(dfquad_pred(dfq[i], est) > 0.5) for i in idxs]
    tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    acc = accuracy_score(gold, pred)*100; f1 = f1_score(gold, pred, zero_division=0)*100
    print(f"DF-QuAD {lab}: {acc:.1f}%  (F1 {f1:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")

=== GEMMA DF-QuAD on COM2SENSE (200 examples) ===

DF-QuAD 0.5 Base (fair): 69.5%  (F1 65.5)  TN=81 FP=14 FN=47 TP=58
DF-QuAD Est. Base: 74.0%  (F1 72.6)  TN=79 FP=16 FN=36 TP=69


In [15]:
run_dfquad_batch(100)

Have 200/500. Running 100 (Gemma DF-QuAD, rate-limited)...
  10/100 → 210 total
  20/100 → 220 total
  30/100 → 230 total
  40/100 → 240 total
  50/100 → 250 total
  60/100 → 260 total
  70/100 → 270 total
  80/100 → 280 total
  90/100 → 290 total
  100/100 → 300 total
✅ 300/500 done


In [17]:
def dfquad_pred(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

dfq = load_dfq(); idxs = sorted(dfq); gold = [dfq[i]["gold"] for i in idxs]; n = len(idxs)
print(f"=== GEMMA DF-QuAD on COM2SENSE ({n} examples) ===\n")
for est, lab in [(False,"0.5 Base (fair)"), (True,"Est. Base")]:
    pred = [int(dfquad_pred(dfq[i], est) > 0.5) for i in idxs]
    tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    acc = accuracy_score(gold, pred)*100; f1 = f1_score(gold, pred, zero_division=0)*100
    print(f"DF-QuAD {lab}: {acc:.1f}%  (F1 {f1:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")

=== GEMMA DF-QuAD on COM2SENSE (300 examples) ===

DF-QuAD 0.5 Base (fair): 69.3%  (F1 63.8)  TN=127 FP=23 FN=69 TP=81
DF-QuAD Est. Base: 75.7%  (F1 73.5)  TN=126 FP=24 FN=49 TP=101


In [18]:
run_dfquad_batch(100)

Have 300/500. Running 100 (Gemma DF-QuAD, rate-limited)...
  10/100 → 310 total
  20/100 → 320 total
  30/100 → 330 total
  40/100 → 340 total
  50/100 → 350 total
  60/100 → 360 total
  70/100 → 370 total
  80/100 → 380 total
  90/100 → 390 total
  100/100 → 400 total
✅ 400/500 done


In [20]:
def dfquad_pred(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

dfq = load_dfq(); idxs = sorted(dfq); gold = [dfq[i]["gold"] for i in idxs]; n = len(idxs)
print(f"=== GEMMA DF-QuAD on COM2SENSE ({n} examples) ===\n")
for est, lab in [(False,"0.5 Base (fair)"), (True,"Est. Base")]:
    pred = [int(dfquad_pred(dfq[i], est) > 0.5) for i in idxs]
    tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    acc = accuracy_score(gold, pred)*100; f1 = f1_score(gold, pred, zero_division=0)*100
    print(f"DF-QuAD {lab}: {acc:.1f}%  (F1 {f1:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")

=== GEMMA DF-QuAD on COM2SENSE (400 examples) ===

DF-QuAD 0.5 Base (fair): 71.5%  (F1 67.2)  TN=169 FP=26 FN=88 TP=117
DF-QuAD Est. Base: 78.2%  (F1 76.7)  TN=170 FP=25 FN=62 TP=143


In [21]:
run_dfquad_batch(100)

Have 400/500. Running 100 (Gemma DF-QuAD, rate-limited)...
  10/100 → 410 total
  20/100 → 420 total
  30/100 → 430 total
  40/100 → 440 total
  50/100 → 450 total
  60/100 → 460 total
  70/100 → 470 total
  80/100 → 480 total
  90/100 → 490 total
  100/100 → 500 total
✅ 500/500 done


In [23]:
def dfquad_pred(v, est):
    bag = BAG()
    tau = v["topic_base"] if (est and v["topic_base"] is not None) else 0.5
    claim = Argument("db0", v["gold"], float(tau)); bag.arguments["db0"] = claim
    if v["sup"]!="N/A" and v["sup_base"] is not None:
        bag.add_support(Argument("S1","s",float(v["sup_base"])), claim)
    if v["att"]!="N/A" and v["att_base"] is not None:
        bag.add_attack(Argument("A1","a",float(v["att_base"])), claim)
    computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)

dfq = load_dfq(); idxs = sorted(dfq); gold = [dfq[i]["gold"] for i in idxs]; n = len(idxs)
print(f"=== GEMMA DF-QuAD on COM2SENSE — FINAL ({n} examples) ===\n")
for est, lab in [(False,"0.5 Base (fair)"), (True,"Est. Base")]:
    pred = [int(dfquad_pred(dfq[i], est) > 0.5) for i in idxs]
    tn,fp,fn,tp = confusion_matrix(gold, pred, labels=[0,1]).ravel()
    acc = accuracy_score(gold, pred)*100; f1 = f1_score(gold, pred, zero_division=0)*100
    print(f"DF-QuAD {lab}: {acc:.1f}%  (F1 {f1:.1f})  TN={tn} FP={fp} FN={fn} TP={tp}")

=== GEMMA DF-QuAD on COM2SENSE — FINAL (500 examples) ===

DF-QuAD 0.5 Base (fair): 70.6%  (F1 65.1)  TN=216 FP=34 FN=113 TP=137
DF-QuAD Est. Base: 78.6%  (F1 76.6)  TN=218 FP=32 FN=75 TP=175
